# Running Resnet50 Model on Qualcomm AI 100 Device using Evaluator APIs




# Requirements 
- Qualcomm Cloud AI 100 device
- Qualcomm AI 100 Software Development Kit >= 1.6
   - Platform SDK
   - Apps SDK with qaic-pytools installed: use --enable-qaic-pytools flag during the installion process
- Python 3.8*-devel 
     - Centos: yum install python3-devel
     - Ubuntu: apt-get install python3-dev
- Sufficient space (more than 10-15 GB) depending upon the models being used.

### Setting up the Environment

#### Adding Kernel to Jupyter Notebook/Lab 



In [1]:
!/opt/qti-aic/dev/python/qaic-env/bin/python -m pip install ipykernel
!source /opt/qti-aic/dev/python/qaic-env/bin/activate &&  /opt/qti-aic/dev/python/qaic-env/bin/python -m ipykernel install --user --name qaic-env --display-name "Python (qaic-env)";


Installed kernelspec qaic-env in /root/.local/share/jupyter/kernels/qaic-env


#### Note: Restart the Notebook and choose the "Python (qaic-env)" kernel before proceeding further to execute the subsequent cells.


In [2]:
import sys
assert sys.executable  == '/opt/qti-aic/dev/python/qaic-env/bin/python'

#### List out contents of current working directory 
We use this list to cleanup newly created files towards the end of this notebook

In [3]:
import os
files_list = os.listdir()

## Fetching the model and dataset

Dataset for evaluation can be downloaded from [Imagenet](https://image-net.org/download.php) or from  [Kaggle](https://www.kaggle.com/c/imagenet-object-localization-challenge) datasets

we use validation split (~6.3 GB approx) for evaluating the model

Resnet model can be downloaded from [MLCommons Repo](https://github.com/mlcommons/inference/tree/master/vision/classification_and_detection#supported-models)

##### Dataset Preparation:

1. Extract the Images into a folder (Say images)
2. Create a text file with the paths to image file in each line. (Say inputlist.txt)
3. Create a similar text with the paths to the files to be used for calibaration. (Say )
4. Add paths to above files in config yaml file.


#### Standard Preprocessing Steps for Resnet50: 
- Read Image from Disk
- Resize the image to 256x256x3
- Crop the image to 224x224x3
- Normalization (Mean = (123.68,116.78,103.94)

In [4]:
import glob
import os 


# Fetching the resnet50 model 
if not os.path.exists('resnet50_v1.onnx'):
    print("Fetching the resnet50 model")
    !wget https://zenodo.org/record/4735647/files/resnet50_v1.onnx
    print("resnet50 model download complete")
else:
    print("resnet50 model already exists")

dataset_base_path = '/home/ml-datasets/imageNet'
if not os.path.exists(dataset_base_path):
    # Setting up imagenet dataset : Downloads ~6.3 GB approx can take 5-10 minutes based on network speed
    print("Fetching the Imagenet dataset")
    !{sys.executable} utils/setup_imagenet.py -dataset-dir dataset_base_path
    print("Imagenet dataset download complete")
else:
    print("Imagenet dataset folder already exists")


Fetching the resnet50 model
--2023-07-31 09:56:00--  https://zenodo.org/record/4735647/files/resnet50_v1.onnx
Resolving zenodo.org (zenodo.org)... 188.185.124.72
Connecting to zenodo.org (zenodo.org)|188.185.124.72|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 102160076 (97M) [application/octet-stream]
Saving to: ‘resnet50_v1.onnx’

resnet50_v1.onnx    100%[===================>]  97.43M  1.58MB/s    in 2m 41s  

2023-07-31 09:58:43 (621 KB/s) - ‘resnet50_v1.onnx’ saved [102160076/102160076]

resnet50 model download complete
Imagenet dataset folder already exists


### Setting up the Config 


In [5]:
%%writefile resnet50_api_demo.yaml

model:
    globals:
        count: 50
        calib: 5 # -1 implies use all calibration images for PGQ.

    dataset:
        name: ILSVRC2012
        path: '/home/ml-datasets/imageNet'
        inputlist_file: inputlist.txt
        annotation_file: ground_truth.txt
        calibration:
            type: dataset
            file: calibration.txt
        transformations:
            - plugin:
                name: filter_dataset
                params:
                    random: False
                    max_inputs: $count
                    max_calib: $calib


    processing:
        preprocessing:
          transformations:
            - plugin:
                name: resize
                params:
                  dims: 256,256
                  interp: area
                  type: imagenet

            - plugin:
                name: crop
                params:
                  dims: 224,224

            - plugin:
                name: normalize
                params:
                  norm: 1
                  channel_order: RGB
                  means:
                    R: 123.68
                    G: 116.78
                    B: 103.94

            - plugin:
                name: convert_nchw

            - plugin:
                name: create_batch

    inference-engine:
        model_path: 'resnet50_v1.onnx' # if model is not present in current working directory then provide absolute path to the model or path relative to model_zoo_path configured in defaults
        aic_device_ids: [0]  #List of device ids
        platforms:
            - platform:
                name: onnxrt

            - platform:
                name: aic
                precision: int8
                tag: aic_int8
                params:
                    quantization-calibration: Percentile
                    quantization-schema: symmetric_with_uint8
                    percentile-calibration-value: 99.999


        inputs_info:
            - input_tensor_0:
                type: float32
                shape: ['*',3,224,224]

        outputs_info:
            - ArgMax_0:
                type: int64
                shape: ['*']
            - softmax_tensor_0:
                type: float32
                shape: ['*',1001]

    evaluator:
        metrics:
            - plugin:
                 name: topk
                 params:
                     kval: 1,5
                     softmax_index: 1
                     round: 7
                     label_offset: 1


Writing resnet50_api_demo.yaml


In [6]:
from qaic_pytools.api.evaluator import QaicAccuracyEvaluator

# Creation of Evaluator Object 
evaluator = QaicAccuracyEvaluator(model_config='resnet50_api_demo.yaml', disable_console_logger=False, work_dir='API_usage')


/opt/qti-aic/dev/python/qaic-env/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/qti-aic/dev/python/qaic-env/lib/python3.8/site-packages/paramiko/transport.py:236: CryptographyDeprecationWarning: Blowfish has been deprecated
  "class": algorithms.Blowfish,
/opt/qti-aic/dev/python/qaic-env/lib/python3.8/site-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")


INFO: Loading model config
INFO: Using Memory Pipeline for accuracy evaluation
INFO: Model Loaded Successfully


In [7]:
# Evaluate the model
status= evaluator.evaluate(device_id=0) 

INFO: Executing dataset plugins
INFO: Using devices: [0] for evaluation
INFO: Total platform configurations: 2
INFO: Total inputs for execution: 50 and calibration: 5
INFO: Approximate disk usage  - 31.96 MB
INFO: Cleaning up model..
INFO: Executing Preprocessors for calibration inputs
INFO: Generating PGQ profile
INFO: PGQ profile Generation Completed
INFO: Starting compilation for plat1_aic
INFO: Executing plat0_onnxrt on CPU
INFO: Currently Running [1] compilations in parallel
INFO: Starting Inference for: plat0_onnxrt
INFO: Metrics for plat0_onnxrt : {'top1': 0.8, 'top5': 0.88, 'count': 50}
INFO: Completed Inference for : plat0_onnxrt
INFO: Waiting for compilations to complete. Completed: [1/2].
INFO: Waiting for compilations to complete. Completed: [1/2].
INFO: Waiting for compilations to complete. Completed: [1/2].
INFO: Waiting for compilations to complete. Completed: [1/2].
INFO: Completed compilation for plat1_aic took: 87.23819899559021 seconds
INFO: Executing plat1_aic on AI

### List all configured Platforms:  


In [8]:
evaluator.list_all_platforms()

INFO: 
  Platform Config Idx  Platform Type    Platform Tag(s)    Platform Params
---------------------  ---------------  -----------------  -----------------------------------------
                    0  onnxrt
                    1  aic              aic_int8           quantization-calibration:Percentile
                                                           quantization-schema:symmetric_with_uint8
                                                           percentile-calibration-value:99.999


### Example: Update Exisitng Platforms:  

Below shows an example usage, we are updating the percentile-calibration-value from  99.999 to 99.99

In [9]:
evaluator.update_platform_params(platform_tag='aic_int8',params={ "percentile-calibration-value": 99.99})
evaluator.list_all_platforms()



INFO: Platforms with 'aic_int8' tag has been updated 
INFO: 
  Platform Config Idx  Platform Type    Platform Tag(s)    Platform Params
---------------------  ---------------  -----------------  -----------------------------------------
                    0  onnxrt
                    1  aic              aic_int8           quantization-calibration:Percentile
                                                           quantization-schema:symmetric_with_uint8
                                                           percentile-calibration-value:99.99


In [10]:
evaluator.manager.config._inference_config._platforms

[name: onnxrt, self._tag=[''] params: {} , self._precision='fp32' self._use_precompiled=None self._reuse_pgq=True self._is_ref=False self._gpu_ids=[],
 name: aic, self._tag=['aic_int8'] params: {'quantization-calibration': 'Percentile', 'quantization-schema': 'symmetric_with_uint8', 'percentile-calibration-value': '99.99'} , self._precision='int8' self._use_precompiled=None self._reuse_pgq=True self._is_ref=False self._gpu_ids=[] self._binary_path='API_usage/infer/plat1_aic/temp']

### Adding Platform

In [11]:
evaluator.add_platform('aic', precision='fp16',tag='aic_fp16')


INFO: Platform aic [aic_fp16] added Successfully


In [12]:
# List all platforms 
evaluator.list_all_platforms()


INFO: 
  Platform Config Idx  Platform Type    Platform Tag(s)    Platform Params
---------------------  ---------------  -----------------  -----------------------------------------
                    0  onnxrt
                    1  aic              aic_int8           quantization-calibration:Percentile
                                                           quantization-schema:symmetric_with_uint8
                                                           percentile-calibration-value:99.99
                    3  aic              aic_fp16


In [13]:
# Evaluating only fp16 Platform with performance scores.
status= evaluator.evaluate(device_id=[0],platform_tag='aic_fp16',enable_perf_flag=True,perf_iter_count=100) 

INFO: Executing dataset plugins
INFO: Using devices: [0] for evaluation
INFO: Total platform configurations: 1
INFO: Total inputs for execution: 50 and calibration: 0
INFO: Approximate disk usage  - 28.9 MB
INFO: Cleaning up model..
INFO: Defaulting to realtime preprocessing as only single platforms is being executed. use dump_outputs feature to write intermediate stage outputs into disk
INFO: Starting compilation for plat0_aic
INFO: Waiting for compilations to complete. Completed: [0/1].
INFO: Waiting for compilations to complete. Completed: [0/1].
INFO: Completed compilation for plat0_aic took: 80.09975838661194 seconds
INFO: Executing plat0_aic on AIC device: 0
INFO: Starting Inference for: plat0_aic
INFO: Metrics for plat0_aic : {'top1': 0.8, 'top5': 0.88, 'count': 50}
INFO: Completed Inference for : plat0_aic
Platform    Status    Precision    Params    Metrics      Comparator      Throughput(Inf/Sec)    Latency(us)
----------  --------  -----------  --------  -----------  -------

#### Cleanup (Optional)
Remove newly created files by this notebook

In [15]:
import os
import shutil
from pathlib import Path

new_files_list = os.listdir()
remove_files = list(set(new_files_list) - set(files_list))

# Add any intermediate files generated in different paths other than current work directory
# remove_files.append(<path>)

for file in remove_files:
    file = Path(file)
    if not file.exists():
        continue
    if file.is_file():
        file.unlink(missing_ok=True)
    else:
        shutil.rmtree(file, ignore_errors=True)
print("Cleanup finished")

Cleanup finished
